# Il mondo come grafo

Il codice del capitolo [«Il mondo come grafo»](https://book.paithon.it/main/GraphNeuralNetwork/dati-a-grafo.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.%pip install -q numpy torch torch-geometric torchvision

## Il mondo come grafo

[Leggi la pagina](https://book.paithon.it/main/GraphNeuralNetwork/dati-a-grafo.html)


### Un esempio numerico: dalla figura alla matrice


In [ ]:
import numpy as npA = np.array([          # matrice di adiacenza (righe/colonne = nodi 1..5)    [0, 1, 1, 0, 0],    [1, 0, 1, 0, 0],    [1, 1, 0, 1, 0],    [0, 0, 1, 0, 1],    [0, 0, 0, 1, 0],])gradi = A.sum(axis=1)          # somma di ogni riga -> [2 2 3 2 1]D = np.diag(gradi)             # matrice dei gradi (diagonale)A_tilde = A + np.eye(5, dtype=int)   # aggiunge i self-loop: A + Iprint(gradi)                   # [2 2 3 2 1]print(A_tilde.sum(axis=1))     # gradi con i cappi: [3 3 4 3 2]

## Message passing: il cuore delle GNN

[Leggi la pagina](https://book.paithon.it/main/GraphNeuralNetwork/message-passing.html)


### Un layer GCN in PyTorch


In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Fclass GCNLayer(nn.Module):    def __init__(self, in_dim, out_dim):        super().__init__()        self.lin = nn.Linear(in_dim, out_dim, bias=False)  # la matrice W    def forward(self, H, A_hat):        # H: (N, in_dim) stati dei nodi; A_hat: (N, N) adiacenza normalizzata        return A_hat @ self.lin(H)  # Â (H W)class GCN(nn.Module):    def __init__(self, in_dim, hid, n_classi):        super().__init__()        self.gc1 = GCNLayer(in_dim, hid)        self.gc2 = GCNLayer(hid, n_classi)    def forward(self, H, A_hat):        H = F.relu(self.gc1(H, A_hat))  # primo strato + ReLU        H = self.gc2(H, A_hat)          # secondo strato: logit per nodo        return H

*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

model = GCN(in_dim=1433, hid=16, n_classi=7)
opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

for epoca in range(200):
    model.train()
    opt.zero_grad()
    logit = model(H, A_hat)                                # tutti i nodi
    loss = F.cross_entropy(logit[mask_train], y[mask_train])  # solo etichettati
    loss.backward()                                        # backprop su tutto il grafo
    opt.step()
```


In [ ]:
from torch_geometric.nn import GCNConvconv = GCNConv(in_channels=1433, out_channels=16)# forward: conv(x, edge_index), con x di forma (N, 1433)

## Oltre la GCN: GraphSAGE, GAT e applicazioni

[Leggi la pagina](https://book.paithon.it/main/GraphNeuralNetwork/architetture-applicazioni.html)


### GraphSAGE: imparare a generalizzare


In [ ]:
import torchfrom torch import nnfrom torch_geometric.nn import SAGEConvclass GraphSAGE(nn.Module):    def __init__(self, in_dim, hid_dim, out_dim):        super().__init__()        # aggr='mean' e' l'aggregatore di default; 'max' -> pooling        self.conv1 = SAGEConv(in_dim, hid_dim, aggr="mean")        self.conv2 = SAGEConv(hid_dim, out_dim, aggr="mean")    def forward(self, x, edge_index):        x = torch.relu(self.conv1(x, edge_index))  # primo giro di vicinato        return self.conv2(x, edge_index)           # secondo giro

### GAT: non tutti i vicini contano uguale


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

from torch_geometric.nn import GATConv

# 8 teste concatenate: l'uscita ha dimensione hid_dim * 8
conv1 = GATConv(in_dim, hid_dim, heads=8, concat=True)
# strato finale: le teste si mediano invece di concatenarsi
conv2 = GATConv(hid_dim * 8, out_dim, heads=1, concat=False)
```
